# Universe applied test

This notebook shows the simplified final structure applied directly.

Key idea:
- `Universe` is the edited main object
- it behaves like your earlier `Portfolio`
- but now it can store multiple constructions on the same loaded dataset

In [1]:
from portafolios import Universe, local_loader, EqualWeightConstructor, NaiveRiskParity, Markowitz

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "data_clean_stock_data.csv"
YF_SNAPSHOT_PATH = PROJECT_ROOT / "data" / "yf_snapshot.csv"

print("Project root:", PROJECT_ROOT)
print("Processed CSV exists:", CSV_PATH.exists())
print("Yahoo snapshot exists:", YF_SNAPSHOT_PATH.exists())


Project root: C:\Users\narro\Desktop\semestre\honores_actuaria
Processed CSV exists: True
Yahoo snapshot exists: True


## 1. Same creation pattern as before

In [2]:
u = Universe(
    tickers=["AAPL", "MSFT", "AMZN", "GOOG"],
    start="2020-01-01",
    end="2020-06-30",
    loader=local_loader,
    loader_kwargs={"path": CSV_PATH, "prefer_adj_close": True},
    freq="B",
).preparar_datos()

u.prices.head()


,AAPL,MSFT,AMZN,GOOG
Date,,,,
2020-01-02,72.776606,153.428246,94.900497,68.186813
2020-01-03,72.771729,152.683705,94.309998,68.439404
2020-01-06,72.621654,151.872308,95.184502,69.663459
2020-01-07,72.849231,152.416407,95.694504,69.921535
2020-01-08,73.706271,153.495104,95.550003,70.337515


In [3]:
print("Tickers:", u.tickers)
print("Prices shape:", u.prices.shape)
print("Returns shape:", u.returns.shape)
print("Metadata:", u.metadata)


Tickers: ['AAPL', 'MSFT', 'AMZN', 'GOOG']
Prices shape: (129, 4)
Returns shape: (128, 4)
Metadata: {'source': 'callable_dataframe', 'frequency': 'B', 'universe_name': 'universe', 'output_dir': 'outputs\\runs\\universe'}


## 2. Apply multiple constructions on the same object

In [4]:
u.construir(EqualWeightConstructor(), label="equal_weight_demo", ann_factor=252)
u.construir(NaiveRiskParity(), label="naive_demo", ann_factor=252)
u.construir(Markowitz(), label="markowitz_demo", ann_factor=252, ret_kind="simple", allow_short=False)

u.list_constructions()


['equal_weight_demo', 'naive_demo', 'markowitz_demo']

## 3. Active construction still behaves like before

In [5]:
print("Active label:", u.active_label)
print("Current active weights:")
u.weights


Active label: markowitz_demo
Current active weights:


AAPL    0.000000
AMZN    0.816979
GOOG    0.000000
MSFT    0.183021
dtype: float64

## 4. Stored constructions are now available for comparison

In [6]:
for name in u.list_constructions():
    result = u.get_construction(name)
    print("-", result.name, "|", result.method, "| selected:", result.selected_assets)


- equal_weight_demo | equal_weight | selected: ['AAPL', 'AMZN', 'GOOG', 'MSFT']
- naive_demo | naive_risk_parity | selected: ['AAPL', 'AMZN', 'GOOG', 'MSFT']
- markowitz_demo | markowitz_max_sharpe | selected: ['AMZN', 'MSFT']


In [7]:
markowitz_result = u.get_construction("markowitz_demo")
print("Name:", markowitz_result.name)
print("Method:", markowitz_result.method)
print("Params:", markowitz_result.params)
print("Metrics:", markowitz_result.metrics_insample)
markowitz_result.weights


Name: markowitz_demo
Method: markowitz_max_sharpe
Params: {'ann_factor': 252, 'ret_kind': 'simple', 'allow_short': False}
Metrics: {'n_selected': 2, 'expected_return': 0.7545602733556669, 'volatility': 0.3165896504227456, 'sharpe_m': 2.383401581031137, 'meta_n_assets': 4, 'meta_success': True, 'meta_message': 'Optimization terminated successfully', 'meta_rf_per_period': 0.0, 'meta_objective': 0.15014018709063212, 'meta_allow_short': False, 'meta_ret_kind_used': 'simple'}


AAPL    0.000000
AMZN    0.816979
GOOG    0.000000
MSFT    0.183021
dtype: float64

## 5. In-sample comparison table

In [8]:
comparison = u.compare_insample_metrics()
comparison


,method,display_name,n_selected,expected_return,volatility,sharpe_m,meta_n_assets,meta_nrp_method,meta_nrp_min_vol,meta_nrp_sigma,meta_success,meta_message,meta_rf_per_period,meta_objective,meta_allow_short,meta_ret_kind_used
name,,,,,,,,,,,,,,,,
equal_weight_demo,equal_weight,Equal Weight,4,0.486218,0.302358,1.608084,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
markowitz_demo,markowitz_max_sharpe,Markowitz Max Sharpe,2,0.754560,0.316590,2.383402,4.0,NaN,NaN,NaN,True,Optimization terminated successfully,0.0,0.15014,False,simple
naive_demo,naive_risk_parity,Naive Risk Parity (1/sigma),4,0.482342,0.301091,1.601981,NaN,inverse_vol,1.000000e-08,"{'AAPL': 0.024012558289591562, 'MSFT': 0.02147...",NaN,NaN,NaN,NaN,NaN,NaN


## 6. Duplicate labels are blocked

In [9]:
try:
    u.construir(EqualWeightConstructor(), label="equal_weight_demo")
except Exception as exc:
    print(type(exc).__name__, str(exc))


ValueError Ya existe una construccion con el nombre 'equal_weight_demo'.
